# **GOES 8-19 X-Ray Sensor (XRS) AI-Ready Data Summary and Analysis**

Creators: James Mothersbaugh III <sup>1,2</sup>, Janet Machol <sup>1,2</sup>, Elysia Lucas <sup>1,2</sup>, Ann Marie Mahon <sup>1,2</sup>
* Affiliations: <sup>1</sup> University of Colorado CIRES and <sup>2</sup> NOAA National Centers for Environmental Information

Contact: james.mothersbaugh@noaa.gov, janet.machol@noaa.gov, elysia.lucas@noaa.gov, ann.mahon@noaa.gov

Revision History:
   * 18 May 2026: Version 1
   * 27 May 2026: Version 2
   * 16 July 2026: Version 3

Acknowledgements: \
Funding support was provided by the Big Earth Data Initiative (BEDI) and the NOAA Center for AI (NCAI) \
Rob Redmon, Chris Slocum and Douglas Rao provided review and editing support

## **Definitions**

**Angstrom** \
10<sup>-10</sup> meters. This unit is used to measure wavelengths.

**Coronal Mass Ejection (CME)** \
A large ejection of plasma from the Sun's corona. Forecasters classify these ejections on specific severity scales to predict space weather impacts on Earth.

**Data Quality Flag** \
A number set in the data variables that indicate if a spacecraft, instrument or celestial event occurred while irradiance was measured. The number corresponding to each data quality flag are bit numbers.

**Geostationary Orbit** \
An orbit in which a satellite orbits the Earth at the same speed the Earth rotates. This orbit allows the satellite to remain in a fixed location relative to the Earth.

**Irradiance** \
The power or intensity of solar energy hitting a specific area, measured in Watts per square meter. 'Peak irradiance' is the maximum power reached during a solar flare, while 'background irradiance' is the normal, ambient energy from the Sun.

**NetCDF** \
The data file format used to store XRS solar measurements. These files can be read using Python libraries. The full name of the file format is network Common Data Form.

**Solar Flare** \
A massive eruption of charged particles and strong magnetic fields from the Sun, traveling at the speed of light.

**UTC** \
Coordinated Universal Time. UTC is a common scientific time system, based on Greenwich Mean Time (GMT). Mountain Standard Time is 7.0 hours behind UTC.

## **Summary**

The GOES satellites have provided constant measurements of solar X-ray irradiance in geostationary orbit since 1976. The uses, importance and features of this data set are explained in the text and code below.

Science-quality data is available for GOES 8-19, spanning 1995-2026. The GOES 1-7 science-quality data, spanning 1976-1996, is currently being reprocessed and will be available in late 2026.

## **Prerequisites**

* Basic understanding of the Python programming language
* Python environment in which to run this Jupyter Notebook

## **Learning Outcomes**

This Jupyter notebook demonstrates reading, analyzing, and plotting the GOES 8-19 XRS irradiance and data quality flags.

The notebook uses 3 different data products to explain the science and applications of the GOES XRS data.
1. 1-minute average irradiance
2. Flare summary irradiance
3. Mission daily average irradiance

After completing this notebook, the user will be able to read, plot, analyze, and understand three of the most useful XRS data products.

Time Estimate
* Estimated text reading time: 10 to 19 min
* Estimated code reading time: 17 to 33 min
* Estimated total reading time: 27 to 52 min

## **Introduction**

The GOES satellites operate in geostationary orbit. The XRS instrument has been present on every GOES satellite, providing over 50 years of continuous observations of X-ray emissions from the Sun. Solar flares are large ejections of material (charged particles such as protons, electrons and strong magnetic fields) from the Sun. The intensity of a solar flare, measured in Watts per square meter at the top of Earth's atmosphere, corresponds to the speed and intensity of the released material.

The NOAA Space Weather Prediction Center (SWPC) uses solar flare data to forecast space weather events and effects on Earth. Solar flares, CMEs and other energetic solar events are classified by severity on the geomagnetic storm, solar radiation storm and radio blackout scales. More information about these scales is available on the SWPC website: https://www.swpc.noaa.gov/noaa-scales-explanation.

The solar cycle is an approximately 11-year time period during which solar flares increase, then decrease, in frequency and intensity. The exact cause of the solar cycle is unknown, but it is well-documented and can be easily seen and studied by analyzing solar flare data. The 50+ year XRS mission provides a calibrated, standardized solar flare data record.

Solar flares are used for a variety of research topics. The causes of the solar cycle, prediction of solar flares, estimating the frequency of catastrophic or impactful flares and methods of coronal heating are all informed by the availability of a long-duration, well-calibrated solar flare data record.

The X-Ray Sensor (XRS) instrument has two channels to measure solar X-ray irradiance. The first channel, called XRS-A, measures the irradiance in the 0.5-4 Angstrom bandpass. The second channel, called XRS-B, measures the irradance at 1-8 Angstroms. The XRS-B channel measures brighter and more energetic irradiances, and is thus used to record and report solar flare data.

The GOES XRS data record is ideal for use in data science projects, and contain a number of features that are AI-ready. The record currently covers 32 years of solar flare data, spanning nearly 3 solar cycles. When the GOES 1-7 data is added, the record will span over 5 decades and 5 solar cycles. This record will contain over 100,000 labeled solar flares, calibrated irradiance measurements and precise data quality flagging.

## **Tutorial**

### Software
The libraries/packages below must be imported to the Jupyter Notebook. These are used to read the XRS NetCDF data files, and to manipulate the data in the files. Some syntax in this notebook is Python version-dependent, and some software libraries may need to be updated to run in the notebook.

In [ ]:
import subprocess, sys

PACKAGES = {"netCDF4": "netCDF4>=1.7.2", "ncflag": "ncflag>=0.3.3", "numpy": "numpy>=2.2.4", "cftime": "cftime>=1.6.4.post1", \
            "matplotlib": "matplotlib>=3.10.1", "requests": "requests>=2.32.3", "bs4": None}

# Loop through each package to either import or install.
for package in PACKAGES:
    try:
        __import__(package)
        # First, attempt to import the package.
    except ImportError:
        # If package import is unsuccessful, install using pip.
        # The command structure is <python executable> -m pip install <package_name>.
        package_name = package if PACKAGES[package] is None else PACKAGES[package]
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])
        # Make another attempt to import the package.
        __import__(package)

In [ ]:
import netCDF4 as nc
import ncflag
# Import libraries to read the NetCDF data files and the data quality flags.

import numpy as np
# Import numpy.

import cftime
import datetime
# Import libraries used to read and plot the data timestamps.

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as tck
# Import matplotlib and specific libraries used for formatting the tick marks on the x-axis of the plots.

import requests
import os
from bs4 import BeautifulSoup
import glob
# Import libraries to search for and download the data files.
# The BeautifulSoup module parses all the content on a particular webpage. It is used to idenfity all the data files on the XRS data
# webpages in the cells below. The data filenames are not static (the date of the data in the file and the version number of the file are
# not static, and can change when the data is reprocessed or updated.

### **Data Acquisition**

Next, the data files that are used in this notebook must be downloaded. The files are downloaded to the current working directory, which is the same directory in which this notebook is currently running.

The files can also be downloaded manually from the NCEI Space Weather data pages at https://www.ncei.noaa.gov/products/goes-r-extreme-ultraviolet-xray-irradiance (GOES 16-19 data; use the arrow-expanding 'Level 2' tab to find the different GOES 16-19 XRS data products) and https://www.ncei.noaa.gov/products/goes-1-15/space-weather-instruments (GOES 8-15 data; go to the 'Solar Irradiance' tab and the 'XRS Science-Quality and Operational Data' drop-down menu to find the different GOES 8-15 XRS data products).

The nomenclature for the filenames used in this notebook is as follows: \
`sci_xrsf-l2-product_gSatNum_date_VersionNumber.nc` \
`product` is the 5-character string indicating the data product (`avg1m` is 1-minute average, `flsum` is flare summary, etc.) \
`SatNum` is the 2-digit string indicating the satellite number (`16`, etc.) \
`VersionNumber` is the 6-character string indicating the data version number (`v1-0-0`, etc.)

The specific files are:
* GOES-12 September 7 2005 1-minute irradiance,
* GOES-12 September 7 2005 flare summary irradiance
* GOES 8-19 mission daily average irradiance (separate file for each satellite).

In [ ]:
xrs_filenames = {}
local_directory = './'
goes_8_15_xrs_data_website = 'https://www.ncei.noaa.gov/data/goes-space-environment-monitor/access/science/xrs/'
goes_16_19_xrs_data_website = 'https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/'
# Data URLs. The GOES 8-15 and GOES 16-19 data files are on different websites.

goes_8_15_xrs_filename_prefixes = ['sci_xrsf-l2-avg1m_g12_d20050907_v', 'sci_xrsf-l2-flsum_g12_d20050907_v', 'sci_xrsf-l2-bkd1d_g08_s', \
                                   'sci_xrsf-l2-bkd1d_g09_s', 'sci_xrsf-l2-bkd1d_g10_s', 'sci_xrsf-l2-bkd1d_g11_s', \
                                   'sci_xrsf-l2-bkd1d_g12_s', 'sci_xrsf-l2-bkd1d_g13_s', 'sci_xrsf-l2-bkd1d_g14_s', \
                                   'sci_xrsf-l2-bkd1d_g15_s']
goes_16_19_xrs_filename_prefixes = ['sci_xrsf-l2-bkd1d_g16_s', 'sci_xrsf-l2-bkd1d_g17_s', 'sci_xrsf-l2-bkd1d_g18_s', \
                                    'sci_xrsf-l2-bkd1d_g19_s']
# Filename prefixes.

for prefix in goes_8_15_xrs_filename_prefixes + goes_16_19_xrs_filename_prefixes:
    local_file = glob.glob(local_directory + prefix + '*.nc')
    if local_file:
        xrs_file = os.path.basename(local_file[0])
        xrs_filenames[prefix] = xrs_file
        print(xrs_file + ' data file exists in the local local working directory')
    else:
        print('A data file matching ' + prefix + '*nc does not exist in the local working directory; downloading file from the GOES ' + \
              'data website')
        satnum = prefix[19:21]
        if prefix == 'sci_xrsf-l2-avg1m_g12_d20050907_v':
            url_string = goes_8_15_xrs_data_website + 'goes12/xrsf-l2-avg1m_science/2005/09/'
            # G12 September 7 2005 1-minute irradiance.
        elif prefix == 'sci_xrsf-l2-flsum_g12_d20050907_v':
            url_string = goes_8_15_xrs_data_website + 'goes12/xrsf-l2-flsum_science/2005/09/'
            # G12 September 7 2005 flare summary irradiance.
        elif prefix in goes_16_19_xrs_filename_prefixes:
            url_string = goes_16_19_xrs_data_website + 'goes' + satnum + '/l2/data/xrsf-l2-bkd1d_science/'
            # G16-19 mission daily average irradiance; each satellite has its own file.
        else:
            url_string = goes_8_15_xrs_data_website + 'goes' + satnum + '/xrsf-l2-bkd1d_science/'
            # G8-15 mission daily average irradiance; each satellite has its own file.
        url = requests.get(url_string)
        if url.status_code == 200:
            url_files = BeautifulSoup(url.text, 'html.parser')
            for link in url_files.find_all('a'):
                xrs_file = link.get('href')
                if xrs_file and xrs_file.startswith(prefix) and xrs_file.endswith('.nc'):
                    with open(local_directory + xrs_file, "wb") as f:
                        r = requests.get(url_string + xrs_file)
                        f.write(r.content)
                        print(xrs_file)
                    xrs_filenames[prefix] = xrs_file
# Check if the required GOES 8-19 XRS data files exist in the local working directory. If a file does not exist in the local working
# directory, it will be downloaded to the local working directory from the GOES 16-19 space weather data website.

In [ ]:
xrs_avg1m_filename = xrs_filenames['sci_xrsf-l2-avg1m_g12_d20050907_v']
xrs_flsum_filename = xrs_filenames['sci_xrsf-l2-flsum_g12_d20050907_v']
g8_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g08_s']
g9_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g09_s']
g10_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g10_s']
g11_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g11_s']
g12_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g12_s']
g13_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g13_s']
g14_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g14_s']
g15_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g15_s']
g16_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g16_s']
g17_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g17_s']
g18_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g18_s']
g19_xrs_avg1d_filename = xrs_filenames['sci_xrsf-l2-bkd1d_g19_s']
# Assign the filenames to individual variables for use in the code below.

<div class="alert alert-block alert-danger">
    <h3>Potential Data Download Issue</h3>

The user may encounter an error when attempting to read data from either the GOES 8-15 or GOES 16-19 data websites. These websites are both part of the NOAA NCEI network, and occasional issues (network security problems, network upgrades, etc.) could cause the data download code above to fail.

In testing, the code would fail at the `url = requests.get(url_string)` lines in the code cell that downloads the data files from the GOES XRS websites. The code may hang on this line, or there could be an error that reads `ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))`.

The data files can be directly downloaded from the GOES XRS data websites if the coded download method does not work. If direct click-and-save download via the website is used, the files must be moved to the same directory in which this Jupyter Notebook is running. Then, the code cell that reads in the data files will still run normally because the files will be in the proper directory.
</div>

## **XRS 1-Minute Data**

The first analysis example uses the XRS 1-minute data. The 1-minute data is an average of the high-cadence 3-second data. The 1-minute data is best used for plotting and analysis on day, month and year time scales.

First, the 1-minute data file is read in. Then, the file header and variable data descriptions are printed out. These printouts contain important information about the contents of the data variables such as array size, values, types, etc.

In [ ]:
xrs_avg1m_data = nc.Dataset(local_directory + xrs_avg1m_filename)
# Read in the XRS 1-minute data file.

print(xrs_avg1m_data)
# Print the 1-minute data file header.

The variables in the NetCDF file are accessed with the following syntax: `file_variable["variable_name"]`. Use this to print the irradiance and quality flag variable summaries for the XRS-A and XRS-B data.

In [ ]:
print('XRS-A irradiance variable')
print(xrs_avg1m_data["xrsa_flux"])
print('')
print('XRS-A data quality flag variable')
print(xrs_avg1m_data["xrsa_flags"])
print('')

print('XRS-B irradiance variable')
print(xrs_avg1m_data["xrsb_flux"])
print('')
print('XRS-B data quality flag variable')
print(xrs_avg1m_data["xrsb_flags"])

The timestamps in the data file are in units of seconds since 12:00:00 UTC on January 1 1970. The Python function `num2pydate`, part of the `cftime` library, converts the timestamps from these units to datetime objects that are in `YYYY-MoMo-DD HH:MinMin:SS` (Year-Month-Day Hour:Minute:Second) format.

In [ ]:
print('Timestamp units: ' + xrs_avg1m_data["time"].units)
xrs_avg1m_timestamps = cftime.num2pydate(xrs_avg1m_data["time"][:], xrs_avg1m_data["time"].units)
# Convert timestamps to datetime objects.

There are 4 quality flags in the data, each with a corresponding bit value. The `xrsa/b_avg1m_flags` object can be indexed to determine the indices at which each flag is set. This then allows for flagged data to be included or excluded from plotting and analysis.

The `ncflag.get_flag()` function is used to find the flags set at each index. The code below loops over each of the values (flag names) in the `xrsa/b_.flag_meanings` array. The `get_flag()` function analyzes the bit values set at each index to determine which flags are set.

A bit value of 0 means only the `good_quality` flag is set. \
A bit value of 1 means only the `eclipsed_by_earth` flag is set. \
A bit value of 6 means the `eclipsed_by_earth` and `saturation` flags are set. \
And so on.

Note: the `temperature_recovery` flag is not set in the GOES 8-12 XRS data. This flag is a historical flag that was used in the real-time data by the NOAA Space Weather Prediction Center (SWPC).

In [ ]:
xrsa_flags = ncflag.FlagWrap.init_from_netcdf(xrs_avg1m_data["xrsa_flags"])
xrsb_flags = ncflag.FlagWrap.init_from_netcdf(xrs_avg1m_data["xrsb_flags"])
# Determine where flags are set.

print('XRS-A Data Quality Flags')
xrsa_avg1m_flags = ncflag.FlagWrap.init_from_netcdf(xrs_avg1m_data["xrsa_flags"])
print(xrsa_avg1m_flags.flag_meanings)
print(xrs_avg1m_data["xrsa_flags"].flag_values)
print('')
print('Number of indices at which the XRS-A quality flags are set:')
for m in xrsa_flags.flag_meanings:
# Loop over all XRSA flags.

    flag_set_a = xrsa_flags.get_flag(m)
    # Determine indices where current flag is and is not set.

    flag_frequency_a = np.count_nonzero(flag_set_a)
    print(f"{m:<20}: {flag_frequency_a}")
    # Print the number of indices at which each flag is set.

print('')
print('XRS-B Data Quality Flags')
xrsb_avg1m_flags = ncflag.FlagWrap.init_from_netcdf(xrs_avg1m_data["xrsb_flags"])
print(xrsb_avg1m_flags.flag_meanings)
print(xrs_avg1m_data["xrsb_flags"].flag_values)
print('')
print('Number of indices at which the XRS-B quality flags are set:')
for n in xrsb_flags.flag_meanings:
# Loop over all XRSB flags.

    flag_set_b = xrsb_flags.get_flag(n)
    # Determine indices where current flag is and is not set.

    flag_frequency_b = np.count_nonzero(flag_set_b)
    print(f"{n:<20}: {flag_frequency_b}")
    # Print the number of indices at which each flag is set.

This day has an eclipse, a saturated flare and some timestamps with bad or missing data in both the XRS-A and XRS-B channels.

The geostationary orbit of the GOES satellites cause 2 eclipse seasons, roughly centered on the spring and fall equinoxes, each year. The spring eclipse season starts at the end of February, peaks in the middle of March and ends in early April. The fall eclipse season starts at the end of August, peaks in the middle of September and ends in early October. As seen by the satellite, the Earth eclipses the Sun each day during the eclipse season. The longest eclipse is 70 minutes.

The loop above can be separated to save arrays of the Boolean values for each flag. Then, the Boolean arrays can be used to determine the timestamps at which each flag is set.

In [ ]:
good_quality_flag_xrsa = xrsa_flags.get_flag('good_data')
eclipse_flag_xrsa = xrsa_flags.get_flag('eclipsed_by_earth')
saturation_xrsa = xrsa_flags.get_flag('saturation')
bad_flag_xrsa = xrsa_flags.get_flag('bad_missing_data')
# Arrays with True/False values indicating the indices at which each XRS-A flag is set and not set.

good_quality_flag_xrsb = xrsb_flags.get_flag('good_data')
eclipse_flag_xrsb = xrsb_flags.get_flag('eclipsed_by_earth')
saturation_xrsb = xrsb_flags.get_flag('saturation')
bad_flag_xrsb = xrsb_flags.get_flag('bad_missing_data')
# Arrays with True/False values indicating the indices at which each XRS-B flag is set and not set.

In [ ]:
good_timestamps_xrsa = xrs_avg1m_timestamps[good_quality_flag_xrsa]
eclipse_timestamps_xrsa = xrs_avg1m_timestamps[eclipse_flag_xrsa]
saturation_timestamps_xrsa = xrs_avg1m_timestamps[saturation_xrsa]
bad_timestamps_xrsa = xrs_avg1m_timestamps[bad_flag_xrsa]
# Extract the timestamps at which each XRS-A flag is set.

print('The XRS-A "eclipsed_by_earth" flag is set at the following timestamps: ' + str(eclipse_timestamps_xrsa[0]) + ' to ' + \
str(eclipse_timestamps_xrsa[-1]))
print('')
print('The XRS-A "saturation" flag is set at the following timestamps: ' + str(saturation_timestamps_xrsa))
print('')

In [ ]:
good_timestamps_xrsb = xrs_avg1m_timestamps[good_quality_flag_xrsb]
eclipse_timestamps_xrsb = xrs_avg1m_timestamps[eclipse_flag_xrsb]
saturation_timestamps_xrsb = xrs_avg1m_timestamps[saturation_xrsb]
bad_timestamps_xrsb= xrs_avg1m_timestamps[bad_flag_xrsb]
# Extract the timestamps at which each XRS-B flag is set.

print('The XRS-B "eclipsed_by_earth" flag is set at the following timestamps: ' + str(eclipse_timestamps_xrsb[0]) + ' to ' + \
str(eclipse_timestamps_xrsb[-1]))
print('')
print('The XRS-B "saturation" flag is set at the following timestamps: ' + str(saturation_timestamps_xrsb))

The eclipse is flagged at the same timestamps in both XRS-A and XRS-B: 04:27-05:31. The `saturation` flag is set at 15 timestamps in XRS-A but only 1 timestamp in XRS-B. This indicates a large flare, but not so large that it saturated XRS-B (which has a higher measurement range) for more than 1 minute. The saturation and eclipse periods will be further explored below.

The `saturation` flag is **rarely** set in the GOES 8-12 XRS data. The XRS detectors on these satellites did not saturate below flares of ~X20 magnitude, and these flares are quite rare (1-2 flares per solar cycle). The saturated flare that is detailed in this notebook was chosen because it provides additional data for analysis and demonstration. More detail about solar flare classification is given in the 'XRS Flare Summary Data' section below.

Now the irradiance can be plotted. The figures below show both the irradiance and the quality flags for a single channel in the same figure. XRS-A and XRS-B are shown in separate figures.

In [ ]:
xrsa_irrad = np.ma.filled(xrs_avg1m_data["xrsa_flux"][:], np.nan)
xrsb_irrad = np.ma.filled(xrs_avg1m_data["xrsb_flux"][:], np.nan)
# Replace irradiance fill & missing values with NaN values in the XRS-A & XRS-B irradiance arrays.

plt.figure(figsize = (30, 20))
for m in xrsa_flags.flag_meanings:
# Loop over all XRSA flags.

    flag_set_a = xrsa_flags.get_flag(m)
    # Determine indices where current flag is and is not set.

    plt.plot(xrs_avg1m_timestamps[flag_set_a], [m for _ in range(len(xrs_avg1m_timestamps[flag_set_a]))], marker = 'x', linestyle = 'none',\
             markersize = 14)
plt.rc('font', size = 25)
plt.xlabel('Hour', fontsize = 30)
plt.ylabel('Data Quality Flags ("x")', fontsize = 30)
plt.gca().xaxis.set_major_locator(mdates.HourLocator())
plt.gca().xaxis.set_minor_locator(mdates.MinuteLocator(30))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H'))
plt.tick_params(axis = 'x', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'x', which = 'minor', length = 8, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 25)
#plt.xlim([datetime.datetime(2005, 9, 7, 17, 0), datetime.datetime(2005, 9, 7, 19, 0)])
# Plot the XRS-A quality flags.
plt.twinx()
plt.plot(xrs_avg1m_timestamps, xrsa_irrad, color = 'black', label = 'XRS-A', marker = '.', markersize = 15, linestyle = 'none')
plt.yscale('log')
plt.ylabel('Irradiance (W/m$^2$)', fontsize = 30)
plt.gca().xaxis.set_major_locator(mdates.HourLocator())
plt.gca().xaxis.set_minor_locator(mdates.MinuteLocator(30))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H'))
plt.tick_params(axis = 'x', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'x', which = 'minor', length = 8, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 30)
lg = plt.legend(loc = 'best', fontsize = 30, numpoints = 1)
lg.legend_handles[0].set_markersize(35)
plt.margins(0.02, 0.05)
plt.title('GOES-12 XRS-A (0.5 - 4 ' + r'$\AA$' + ') 1-Minute Irradiance: ' + xrs_avg1m_timestamps[0].strftime("%Y-%m-%d"), fontsize = 45, \
          y = 1.02)
# Plot the XRS-A irradiance.

plt.figure(figsize = (30, 20))
for n in xrsb_flags.flag_meanings:
# Loop over all XRS-B flags.

    flag_set_b = xrsb_flags.get_flag(n)
    # Determine indices where current flag is and is not set.

    plt.plot(xrs_avg1m_timestamps[flag_set_b], [n for _ in range(len(xrs_avg1m_timestamps[flag_set_b]))], marker = 'x', linestyle = \
             'none', markersize = 14)
plt.rc('font', size = 25)
plt.xlabel('Hour', fontsize = 30)
plt.ylabel('Data Quality Flags ("x")', fontsize = 30)
plt.gca().xaxis.set_major_locator(mdates.HourLocator())
plt.gca().xaxis.set_minor_locator(mdates.MinuteLocator(30))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H'))
plt.tick_params(axis = 'x', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'x', which = 'minor', length = 8, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 30)
#plt.xlim([datetime.datetime(2005, 9, 7, 17, 0), datetime.datetime(2005, 9, 7, 19, 0)])
# Plot the XRS-B quality flags.
plt.twinx()
plt.plot(xrs_avg1m_timestamps, xrsb_irrad, color = 'dodgerblue', label = 'XRS-B', marker = '.', markersize = 15, linestyle = 'none')
plt.yscale('log')
plt.ylabel('Irradiance (W/m$^2$)', fontsize = 30)
plt.gca().xaxis.set_major_locator(mdates.HourLocator())
plt.gca().xaxis.set_minor_locator(mdates.MinuteLocator(30))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H'))
plt.tick_params(axis = 'x', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'x', which = 'minor', length = 8, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 30)
lg = plt.legend(loc = 'best', fontsize = 30, numpoints = 1)
lg.legend_handles[0].set_markersize(35)
plt.margins(0.02, 0.05)
plt.title('GOES-12 XRS-B (1 - 8 ' + r'$\AA$' + ') 1-Minute Irradiance: ' + xrs_avg1m_timestamps[0].strftime("%Y-%m-%d"), fontsize = 45, \
          y = 1.02)
# Plot the XRS-B irradiance.

The irradiance during the eclipse is not shown because the `bad_missing_data` and `eclipsed_by_earth` flags are set during the eclipse. In the NetCDF data file, the `bad_missing_data` corresponds to a fill value in the irradiance arrays. There are two lines in the code cell above that convert the fill values to NaNs, which are not shown on the plot.

<div class="alert alert-block alert-info">
    <h3>User Exercise: Zoom on Flare</h3>

The large flare, which reaches saturation thresholds in both channels, is an interesting feature in the data. The plots can be zoomed in on this flare. Uncomment the `plt.xlim()` lines in the XRS-A and XRS-B code and re-run the cell above.
</div>

The zoomed XRS-A plot clearly shows the flare saturation. Instead of increasing to show the true peak of the flare, the irradiance reaches the limit of the XRS-A detector and flattens until the flare declines below the maximum range of the XRS-A detector.

The zoomed XRS-B plot has only one saturated timestamp. This indicates the flare reached the maximum range of the XRS-B detector, but likely did not increase beyond that irradiance level.

This same solar flare is examined more closely in the next section of the tutorial.

## **XRS Flare Summary Data**

### **Science Introduction**
The GOES XRS data products include files that contain detailed information about solar flares in the X-ray region of the electromagnetic spectrum. Solar flares provide the first indication of impending space weather at Earth, causing auroral lights and impacting critical systems including radio communications, precision navigation, and the power grid.

Solar flares measured by XRS are classified into 5 different classes. In order of increasing energy, these are A, B, C, M and X. The irradiance thresholds (in W/m^2) are, respectively, 1e-8, 1e-7, 1e-6, 1e-5 and 1e-4. The flare summary data reports the peak irradiance of solar flares with both the class and the exact irradiance magnitude, e.g. X2.9 (this would be an X-class flare with a peak irradiance magnitude of 2.9e-4 W/m^2). The largest flares are magnitude X20 or greater and occur 0-1 times during each 11-year solar cycle.

The frequency distribution of the different flares is roughly logarithmic. Low-energy B and C-class flares occur thousands of times over the solar cycle, M-class flares occur several hundred times and X-class flares occur several dozen times. The M and X-class flares are of particular interest to space weather forecasting because those flares are accompanied by particle and magnetic events that can threaten electronic and electromagnetic infrastructure on Earth. The real-time GOES XRS irradiance can be seen in the 'GOES X-RAY FLUX' box in the lower left of the SWPC website: https://www.swpc.noaa.gov/.

### **Data Introduction**
The GOES XRS data processing pipeline includes an algorithm that measures and flags solar flares in real time. The critical information and statistics for each flare are then written to data files for further analysis.

There are two data products containing the solar flare information. The first is the flare detection product. This contains the 1-minute average irradiance and the start, rise, peak, decline and end periods of each solar flare as the algorithm detects the flare in the real-time irradiance. The flare detection product is a preliminary measurement of the flare. It is not possible to determine the magnitude and timestamp of the flare peak until after it has started to decline, so the parts of the flare that are given in the flare detection file are delayed by ~3 minutes.

An additional algorithm is run on the flare detection files that corrects the timestamps of the different flare components and sets them to the actual irradiance measurements of the flare. The correct flare data is available in the flare summary data product. The flare summary data for September 7 2005, the same date analyzed in the 1-minute average example above, is shown below.

Both the flare detection and flare summary data products show only the XRS-B irradiance. The XRS-A irradiance is not used for flare detection because XRS-A is the weak irradiance channel.

The XRS solar flare products constitute a valuable AI-ready data set. The combination 1-minute irradiance and labeled solar flares allow for advanced analysis of the flares relative to the non-flare irradiance.

In [ ]:
xrs_flsum_data = nc.Dataset(local_directory + xrs_flsum_filename)
# Read in the XRS flare summary data file.

xrs_flsum_timestamps = cftime.num2pydate(xrs_flsum_data["time"], xrs_flsum_data["time"].units)
# Set timestamps.

xrsb_flsum_flux = np.ma.filled(xrs_flsum_data["xrsb_flux"][:], np.nan)
# Replace flare summary irradiance fill values with NaN values in the XRS-B flux array.

As with the 1-minute average data, the flare summary data file can be printed to show the header, metadata information and variable names.

The most useful variables in the flare summary data are `status` and `flare_class`. These contain, respectively, the markers indicating the different stages of each flare and the class of the flare (determined by the irradiance magnitude). First, print out these variables to see the length and relevant metadata.

To see the full list of variables in the flare summary, run `print(xrs_flsum_data.variables)` in a code cell. In this example, the variables are printed directly, without examining the entire header first.

In [ ]:
print(xrs_flsum_data["status"])
print('')
print(xrs_flsum_data["flare_class"])

The code below is a simple way to calculate and print the number of flares in each flare class and the total number of flares for this day of data. This provides a concise table that can be used in conjunction with plots of the data, which will be done next.

In [ ]:
flare_start = np.where(xrs_flsum_data["status"][:] == 'EVENT_START')[0]
flare_peak = np.where(xrs_flsum_data["status"][:] == 'EVENT_PEAK')[0]
flare_peak_saturated = np.where(xrs_flsum_data["status"][:] == 'EVENT_PEAK_SATURATED')[0]
flare_end = np.where(xrs_flsum_data["status"][:] == 'EVENT_END')[0]
flare_post = np.where(xrs_flsum_data["status"][:] == 'POST_EVENT')[0]
# Find the indices of the flare summary start, peak, end & post-event.

if len(flare_peak_saturated) > 0:
    xflare_saturated = np.where(xrs_flsum_data["xrsb_flux"][flare_peak_saturated] >= 1e-4)[0]
xflare = np.where(xrsb_flsum_flux[flare_peak] >= 1e-4)[0]
mflare = np.where((xrsb_flsum_flux[flare_peak] < 1e-4) & (xrsb_flsum_flux[flare_peak] >= 1e-5))[0]
cflare = np.where((xrsb_flsum_flux[flare_peak] < 1e-5) & (xrsb_flsum_flux[flare_peak] >= 1e-6))[0]
bflare = np.where((xrsb_flsum_flux[flare_peak] < 1e-6) & (xrsb_flsum_flux[flare_peak] >= 1e-7))[0]
aflare = np.where(xrsb_flsum_flux[flare_peak] < 1e-7)[0]
print('Flare Summary Statistics for GOES-12 ' + xrs_flsum_timestamps[0].strftime("%Y-%m-%d"))
if len(flare_peak_saturated) > 0:
    print("X-class Saturated Flares: " + str(len(xflare_saturated)))
print("X-class Flares: " + str(len(xflare)))
print("M-class Flares: " + str(len(mflare)))
print("C-class Flares: " + str(len(cflare)))
print("B-class Flares: " + str(len(bflare)))
print("A-class Flares: " + str(len(aflare)))
if len(flare_peak_saturated) > 0:
    print("Total flares: " + str(len(flare_peak) + len(flare_peak_saturated)))
else:
    print("Total flares: " + str(len(flare_peak)))
# Number of flares in each class.

Now that the flare statistics have been printed, the 1-minute average irradiance and flare summary status types can be plotted. These offer a visualization of how the flare detection algorithm operates on the 1-minute average irradiance.

In [ ]:
plt.figure(figsize = (30, 20))
aflare_line = np.repeat(1e-8, len(xrs_avg1m_timestamps), axis = 0)
bflare_line = np.repeat(1e-7, len(xrs_avg1m_timestamps), axis = 0)
cflare_line = np.repeat(1e-6, len(xrs_avg1m_timestamps), axis = 0)
mflare_line = np.repeat(1e-5, len(xrs_avg1m_timestamps), axis = 0)
xflare_line = np.repeat(1e-4, len(xrs_avg1m_timestamps), axis = 0)
highflare_line = np.repeat(1e-3, len(xrs_avg1m_timestamps), axis = 0)
plt.plot(xrs_avg1m_timestamps, aflare_line, color = 'black', linewidth = 2, linestyle = 'solid')
plt.plot(xrs_avg1m_timestamps, bflare_line, color = 'black', linewidth = 2, linestyle = 'solid')
plt.plot(xrs_avg1m_timestamps, cflare_line, color = 'black', linewidth = 2, linestyle = 'solid')
plt.plot(xrs_avg1m_timestamps, mflare_line, color = 'black', linewidth = 2, linestyle = 'solid')
plt.plot(xrs_avg1m_timestamps, xflare_line, color = 'black', linewidth = 2, linestyle = 'solid')
plt.plot(xrs_avg1m_timestamps, highflare_line, color = 'black', linewidth = 2, linestyle = 'solid')
plt.plot(xrs_avg1m_timestamps, xrsb_irrad, color = 'dodgerblue', marker = '.', markersize = 15, linestyle = 'none', label = \
         '1-Minute Irradiance')
plt.plot(xrs_flsum_timestamps[flare_start], xrsb_flsum_flux[flare_start], color = 'orange', marker = '.', markersize = 30, linestyle = \
         'none', label = 'Flare Start')
plt.plot(xrs_flsum_timestamps[flare_peak], xrsb_flsum_flux[flare_peak], color = 'red', marker = '.', markersize = 30, linestyle = 'none', \
         label = 'Flare Peak')
if len(flare_peak_saturated > 0):
    plt.plot(xrs_flsum_timestamps[flare_peak_saturated], xrsb_flsum_flux[flare_peak_saturated], color = 'purple', marker = '.', markersize \
             = 30, linestyle = 'none', label = 'Flare Peak Saturated')
plt.plot(xrs_flsum_timestamps[flare_end], xrsb_flsum_flux[flare_end], color = 'lawngreen', marker = '.', markersize = 30, linestyle = \
         'none', label = 'Flare End')
plt.plot(xrs_flsum_timestamps[flare_post], xrsb_flsum_flux[flare_post], color = 'black', marker = '.', markersize = 30, linestyle = 'none',\
         label = 'Flare Post-Event')
plt.yscale('log')
plt.rc('font', size = 30)
plt.xlabel('Hour', fontsize = 30)
plt.ylabel('Irradiance (W/m$^2$)', fontsize = 30)
plt.tick_params(axis = 'x', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'x', which = 'minor', length = 8, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H'))
plt.gca().xaxis.set_major_locator(mdates.HourLocator())
plt.gca().xaxis.set_minor_locator(mdates.MinuteLocator(30))
plt.xticks(fontsize = 30)
plt.yticks(fontsize = 30)
ymin = 0.8 * min(xrsb_irrad)
ymax = 1.2 * max(xrsb_irrad)
plt.ylim([ymin, ymax])
plt.margins(0.02, 0.05)
lg = plt.legend(loc = 'best', fontsize = 28, numpoints = 1)
for i in np.arange(0, 5):
    lg.legend_handles[i].set_markersize(30)
if len(flare_peak_saturated > 0):
    lg.legend_handles[5].set_markersize(30)
plt.title('GOES-12 XRS-B (1 - 8 ' + r'$\AA$' + ') Flare Summary Irradiance: ' + xrs_avg1m_timestamps[0].strftime("%Y-%m-%d"),\
          fontsize = 45, y = 1.02)
flare_mag = np.array([1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3])
flare_label = np.array(['', 'A', 'B', 'C', 'M', 'X', ''])
plt.twinx()
plt.yscale('log')
plt.ylim([ymin, ymax])
flare_range = np.where((flare_mag >= ymin) & (flare_mag <= ymax))
plt.yticks(flare_mag[flare_range], flare_label[flare_range], fontsize = 30)
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.ylabel('Flare Class', fontsize = 30)

The plot above shows the different flare summary statuses marked in the 1-minute average irradiance. The `EVENT_PEAK_SATURATED` status is set in the flare summary data at the peak of the flare. The index (timestamp) of this status corresponds to the `saturation` flag that is set in the 1-minute average data.

<div class="alert alert-block alert-info">
    <h3>User Exercise: Zoom on Flare</h3>

Add code to the cell above that zooms the x-axis time range to 16:30:00-19:00:00, then make the plot that shows this zoomed-in time range. This allows for a closer look at the flare and the flare summary status types.

Answer: `plt.xlim([datetime.datetime(xrs_avg1m_timestamps[0].year, xrs_avg1m_timestamps[0].month, xrs_avg1m_timestamps[0].day, 16, 3, 0), datetime.datetime(xrs_avg1m_timestamps[0].year, xrs_avg1m_timestamps[0].month, xrs_avg1m_timestamps[0].day, 19, 0, 0)])` \
Add this line below the `plt.ylim()` lines (lies 42 & 55) in the plotting code in the cell above.
</div>

As a final analysis of the flare summary data, the class of each flare can be printed. The existing arrays defining the flare peaks can be utilized for this. The saturated flare is an X24.4 flare, which is much stronger than the usual X-class flares.

In [ ]:
print('Flare Peak Classes')
print(xrs_flsum_data["flare_class"][flare_peak])
print(xrs_flsum_data["flare_class"][flare_peak_saturated])

## **XRS Daily Average Data**

The GOES 8-19 XRS data includes a product with the daily average and daily background irradiance. This product is most useful for analyzing the data on long-term scales of years to decades. The 1-minute average data product is used to create the daily average. The daily average products are available as mission-length data files.

The best way to analyze the daily average data is to plot the mission-length data for multiple satellites. On long-term timescales, the solar cycle is visible in the data. Some of the GOES XRS instruments span most of a solar cycle.

First, the data files must be read in to Python. Then the timestamps can be set.

In [ ]:
g8_xrs_avg1d_data = nc.Dataset(local_directory + g8_xrs_avg1d_filename)
g9_xrs_avg1d_data = nc.Dataset(local_directory + g9_xrs_avg1d_filename)
g10_xrs_avg1d_data = nc.Dataset(local_directory + g10_xrs_avg1d_filename)
g11_xrs_avg1d_data = nc.Dataset(local_directory + g11_xrs_avg1d_filename)
g12_xrs_avg1d_data = nc.Dataset(local_directory + g12_xrs_avg1d_filename)
g13_xrs_avg1d_data = nc.Dataset(local_directory + g13_xrs_avg1d_filename)
g14_xrs_avg1d_data = nc.Dataset(local_directory + g14_xrs_avg1d_filename)
g15_xrs_avg1d_data = nc.Dataset(local_directory + g15_xrs_avg1d_filename)
g16_xrs_avg1d_data = nc.Dataset(local_directory + g16_xrs_avg1d_filename)
g17_xrs_avg1d_data = nc.Dataset(local_directory + g17_xrs_avg1d_filename)
g18_xrs_avg1d_data = nc.Dataset(local_directory + g18_xrs_avg1d_filename)
g19_xrs_avg1d_data = nc.Dataset(local_directory + g19_xrs_avg1d_filename)
# Read in the XRS daily average data files.

In [ ]:
goes8_timestamps = cftime.num2pydate(g8_xrs_avg1d_data["time"][:], g8_xrs_avg1d_data["time"].units)
goes9_timestamps = cftime.num2pydate(g9_xrs_avg1d_data["time"][:], g9_xrs_avg1d_data["time"].units)
goes10_timestamps = cftime.num2pydate(g10_xrs_avg1d_data["time"][:], g10_xrs_avg1d_data["time"].units)
goes11_timestamps = cftime.num2pydate(g11_xrs_avg1d_data["time"][:], g11_xrs_avg1d_data["time"].units)
goes12_timestamps = cftime.num2pydate(g12_xrs_avg1d_data["time"][:], g12_xrs_avg1d_data["time"].units)
goes13_timestamps = cftime.num2pydate(g13_xrs_avg1d_data["time"][:], g13_xrs_avg1d_data["time"].units)
goes14_timestamps = cftime.num2pydate(g14_xrs_avg1d_data["time"][:], g14_xrs_avg1d_data["time"].units)
goes15_timestamps = cftime.num2pydate(g15_xrs_avg1d_data["time"][:], g15_xrs_avg1d_data["time"].units)
goes16_timestamps = cftime.num2pydate(g16_xrs_avg1d_data["time"][:], g16_xrs_avg1d_data["time"].units)
goes17_timestamps = cftime.num2pydate(g17_xrs_avg1d_data["time"][:], g17_xrs_avg1d_data["time"].units)
goes18_timestamps = cftime.num2pydate(g18_xrs_avg1d_data["time"][:], g18_xrs_avg1d_data["time"].units)
goes19_timestamps = cftime.num2pydate(g19_xrs_avg1d_data["time"][:], g19_xrs_avg1d_data["time"].units)
# Convert the timestamps to datetime objects.

The plots of the daily average irradiance below show only the XRS-B irradiance. The code & method used to make this plot can also be used for XRS-A. Additionally, the daily background irradiance can be plotted and analyzed in the same way as the daily background irradiance.

The plotting code below creates a timestamp array by concatenating the timestamp arrays of different satellites together. This is used to plot the flare class lines. The irradiance for each individual satellite is plotted with its own timestamps.

In [ ]:
flare_timestamps = np.concatenate((goes8_timestamps, goes10_timestamps, goes14_timestamps, goes16_timestamps, goes19_timestamps), \
                                  axis = None)
# Time range for the flare class lines.

zeroflare_line = np.repeat(1e-9, len(flare_timestamps), axis = 0)
aflare_line = np.repeat(1e-8, len(flare_timestamps), axis = 0)
bflare_line = np.repeat(1e-7, len(flare_timestamps), axis = 0)
cflare_line = np.repeat(1e-6, len(flare_timestamps), axis = 0)
mflare_line = np.repeat(1e-5, len(flare_timestamps), axis = 0)
xflare_line = np.repeat(1e-4, len(flare_timestamps), axis = 0)
flare_lines = np.array([zeroflare_line, aflare_line, bflare_line, cflare_line, mflare_line, xflare_line])
# Define the flare class lines for the plot.

goes8_xrsb_avg_irrad = np.ma.filled(g8_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes9_xrsb_avg_irrad = np.ma.filled(g9_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes10_xrsb_avg_irrad = np.ma.filled(g10_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes11_xrsb_avg_irrad = np.ma.filled(g11_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes12_xrsb_avg_irrad = np.ma.filled(g12_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes13_xrsb_avg_irrad = np.ma.filled(g13_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes14_xrsb_avg_irrad = np.ma.filled(g14_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes15_xrsb_avg_irrad = np.ma.filled(g15_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes16_xrsb_avg_irrad = np.ma.filled(g16_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes17_xrsb_avg_irrad = np.ma.filled(g17_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes18_xrsb_avg_irrad = np.ma.filled(g18_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
goes19_xrsb_avg_irrad = np.ma.filled(g19_xrs_avg1d_data["avg1d_xrsb_flux"][:], np.nan)
# Replace irradiance fill values with NaN values in the XRSB daily average irradiance array for each satellite.

In [ ]:
plt.figure(figsize = (30, 20))
for i in np.arange(0, len(flare_lines)):
    plt.plot(flare_timestamps, flare_lines[i], color = 'black', linewidth = 3, linestyle = 'solid')
plt.plot(goes8_timestamps, goes8_xrsb_avg_irrad, color = 'blue', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-08')
plt.plot(goes9_timestamps, goes9_xrsb_avg_irrad, color = 'mediumorchid', marker = '.', markersize = 11, linestyle = 'none', label = \
         'GOES-09')
plt.plot(goes10_timestamps, goes10_xrsb_avg_irrad, color = 'orange', marker = '.', markersize = 11, linestyle = 'none', label = \
         'GOES-10')
plt.plot(goes11_timestamps, goes11_xrsb_avg_irrad, color = 'red', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-11')
plt.plot(goes12_timestamps, goes12_xrsb_avg_irrad, color = 'green', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-12')
plt.plot(goes13_timestamps, goes13_xrsb_avg_irrad, color = 'brown', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-13')
plt.plot(goes14_timestamps, goes14_xrsb_avg_irrad, color = 'hotpink', marker = '.', markersize = 11, linestyle = 'none', label = \
         'GOES-14')
plt.plot(goes15_timestamps, goes15_xrsb_avg_irrad, color = 'darkgrey', marker = '.', markersize = 11, linestyle = 'none', label = \
         'GOES-15')
plt.plot(goes16_timestamps, goes16_xrsb_avg_irrad, color = 'purple', marker = '.', markersize = 11, linestyle = 'none', label = \
         'GOES-16')
plt.plot(goes17_timestamps, goes17_xrsb_avg_irrad, color = 'cyan', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-17')
plt.plot(goes18_timestamps, goes18_xrsb_avg_irrad, color = 'lime', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-18')
plt.plot(goes19_timestamps, goes19_xrsb_avg_irrad, color = 'black', marker = '.', markersize = 11, linestyle = 'none', label = 'GOES-19')
plt.yscale('log')
plt.rc('font', size = 22)
plt.xlabel('Year', fontsize = 35)
plt.ylabel('XRS-B Irradiance (W/m$^2$)', fontsize = 35)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_minor_locator(mdates.MonthLocator(7))
plt.tick_params(axis = 'x', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'x', which = 'minor', length = 8, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.xticks(fontsize = 25, rotation = 70, ha = 'center', va = 'top')
plt.yticks(fontsize = 30)
lg = plt.legend(loc = 'upper left', fontsize = 23, numpoints = 1, bbox_to_anchor = (1.02, 1))
for i in np.arange(0, 12):
    lg.legend_handles[i].set_markersize(35)
plt.margins(0.02, 0.05)
plt.ylim([6e-10, 2e-4])
plt.title('GOES XRS-B (1 - 8 ' + r'$\AA$' + ') Daily Average Irradiance', fontsize = 45, y = 1.02)
# Plot XRSB daily average irradiance.
yflaremag = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4]
yflarelabel = ['', 'A', 'B', 'C', 'M', 'X']
plt.twinx()
plt.yscale('log')
plt.ylim([6e-10, 2e-4])
plt.yticks(yflaremag, yflarelabel, fontsize = 30)
plt.tick_params(axis = 'y', which = 'major', length = 18, width = 2, direction = 'in')
plt.tick_params(axis = 'y', which = 'minor', length = 8, width = 2, direction = 'in')
plt.ylabel('Flare Class', fontsize = 35)
# Label the right y-axis with the flare class levels.

The XRS-B daily average irradiance plot shows a great deal of information. The key features are explained below.

The GOES 8-19 data spans nearly 3 solar cycles. These cycles reach solar maximum in 2001-2002, 2014-2015 and 2025. The solar minimum periods are in 1996-1997, 2007-2009 and 2019-2020. The daily average irradiance is below the A-class flare level during the solar minimum periods and equivalent to weak M-class flares during the solar maximum periods. The largest daily average values are affected by very large (and, in the case of GOES 8-12, sometimes saturated) X-class flares on those days. These flares are visible in April 2001, October/November 2003, September 2005 and September 2017.

There are 3 series of XRS instruments shown on this plot: GOES 8-12 (1995-2009), GOES 13-15 (2010-2019) and GOES 16-19 (2017-present). Each satellite series spans approximately 1 solar cycle. The plot shows excellent agreement among the satellites within each series, and through each series' solar cycle.

The irradiance values at solar maximum are consistent across all 3 satellite series/solar cycles. The solar minima are not as consistent. The first 2 minima, measured by the GOES 8-12 satellites, have a noticeably different minimum irradiance measurement than the solar minima around 2019-2020 (measured by GOES 14-17). This indicates the difference in the range of the XRS instruments on the older satellites, the limitations of reprocessing with old data and possible other unknown instrument issues.

<div class="alert alert-block alert-success">
    <h3>User Exercise</h3>
    
Create a histogram plot showing the distribution of all flares, using the flare classes as the histogram bins, for the GOES 8-12 data. The GOES 8-12 data spans a full solar cycle, and is thus representative of the distribution of the different flare classes across a solar cycle.

In this exercise above, use the following code syntax that is already shown in this tutorial:
- Plotting syntax for the flare classes
- x-axis and y-axis labeling/formatting from the plot of the daily average data
- Flare status identification and indexing in the flare summary section

The mission-length flare summary data files for GOES 8-12 need to be downloaded from the GOES 8-15 XRS data website. These files are sufficient to make a histogram plot of the flares.
</div>

In [ ]:
xrs_avg1m_data.close()
xrs_flsum_data.close()
g8_xrs_avg1d_data.close()
g9_xrs_avg1d_data.close()
g10_xrs_avg1d_data.close()
g11_xrs_avg1d_data.close()
g12_xrs_avg1d_data.close()
g13_xrs_avg1d_data.close()
g14_xrs_avg1d_data.close()
g15_xrs_avg1d_data.close()
g16_xrs_avg1d_data.close()
g17_xrs_avg1d_data.close()
g18_xrs_avg1d_data.close()
g19_xrs_avg1d_data.close()
# Close the open NetCDF files.

## **Data Statement**

GOES 16-19 XRS data is available at https://www.ncei.noaa.gov/products/goes-r-extreme-ultraviolet-xray-irradiance. \
GOES 1-15 XRS data is available under the 'Solar Irradiance' tab at https://www.ncei.noaa.gov/products/goes-1-15/space-weather-instruments.

GOES 16-19 XRS data documentation is available under the 'Documentation' tab at https://www.ncei.noaa.gov/products/goes-r-extreme-ultraviolet-xray-irradiance. \
GOES 1-15 XRS data documentation is available at https://www.ncei.noaa.gov/products/goes-r-extreme-ultraviolet-xray-irradiance.

## **Data Uses**

The GOES 16-19 XRS data includes measurements of the location of the solar flares on the Sun's surface. This flare location data allows for cross-calibration with other solar flare location measurements (e.g., the GOES SUVI instrument) and analysis of both the flare irradiance and its location on the Sun.

The SWPC forecast office uses the real-time XRS irradiance, flare summary and flare location data to forecast space weather events and their effects on Earth.

Solar flares occur when magnetic field lines tangle and break on the Sun's photosphere. The magnetic field behavior is visible on the photosphere as a darkened area called a sunspot. The number of sunspots, and thus the frequency and intensity of solar flares, increases and decreases in tandem with the solar cycle. A solar flare record that includes multiple solar cycles allows for research of the causes of the solar cycle and long-term sunspot behavior.

Solar flares can have an effect on the temperature of the Sun's corona. Long-term solar flare data allows for modeling and research of the heating mechanisms of flares and how they affect the outer regions of the Sun.

## **Recent Data Publications**

1. GOES-R Series X-Ray Sensor (XRS) 2: On-Orbit Measurements and Calibrations. Machol et. al, 2026; https://doi.org/10.1029/2026JA035181
2. The Thermal Properties of Solar Flares over Three Solar Cycles Using GOES X-Ray Observations. Ryan et. al, 2012; DOI 10.1088/0067-0049/202/2/11
3. The Greatest GOES Soft X-ray Flares: Saturation and Recalibration over Two Hale Cycles. Hudson et. al, 2024; DOI 10.1007/s11207-024-02287-x
4. GOES-R Series X-Ray Sensor (XRS): 1. Design and Pre-Flight Calibration. Woods et. al, 2024; https://doi.org/10.1029/2024JA032925
5. How Good Are GOES XRS Temperatures and Emission Measures? White et. al, 2026; https://doi.org/10.1007/s11207-026/02618-0

## **References**

Sections of this code were adapted from the EXIS Python example scripts available at https://cires-stp.github.io/goesr-spwx-examples/examples/index.html#exis.

## **Metadata**
- Packages
     - Python
     - NetCDF
     - Matplotlib
- Domain
    - NOAA
- Application keywords
    - Data quality
- Geophysical keywords
    - Solar irradiance
    - Solar flares
    - Space weather
- AI keywords
    - AI-ready data
    - Labeled data

## **Disclaimer**

This Jupyter notebook is a scientific product and is not official communication of the National Oceanic and Atmospheric Administration, or the United States Department of Commerce. All NOAA Jupyter notebooks are provided on an 'as is' basis and the user assumes responsibility for its use. Any claims against the Department of Commerce or Department of Commerce bureaus stemming from the use of this Jupyter notebook will be governed by all applicable Federal law. Any reference to specific commercial products, processes, or services by service mark, trademark, manufacturer, or otherwise does not constitute or imply their endorsement, recommendation or favoring by the Department of Commerce. The Department of Commerce seal and logo, or the seal and logo of a DOC bureau, shall not be used in any manner to imply endorsement of any commercial product or activity by DOC or the United States Government.